# 03. Model Training & Cross-Validation

**ColoGrowth-ML: Cross-Platform Calibration Benchmark for Colon Cancer Proliferation Classification**

Train 4 classifiers to predict High vs Low proliferation class from gene expression + clinical features.

### Nested Cross-Validation with StabilitySelector
- **StabilitySelector**: bootstrap-based feature selection (B=100 resamples) replaces single-shot SelectKBest for robustness (Meinshausen & Bühlmann, 2010)
- **Inner loop**: hyperparameter tuning + StabilitySelector feature selection inside each fold
- **Outer loop**: unbiased performance estimate via 5-fold CV
- All scaling, variance filtering, and selection inside Pipeline -> zero data leakage

> Full nested CV runs via `python -m src.train --dataset geo` (~10 min).
> This notebook demonstrates model building and loads pre-computed results.

In [ ]:
import sys
sys.path.append('..')
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import roc_auc_score, accuracy_score
from src.stability_selector import StabilitySelector
from src.train import MODEL_BUILDERS
print('Libraries loaded.')

---
## Load Processed Data

In [ ]:
processed_dir = '../data/processed'
if os.path.exists(os.path.join(processed_dir, 'geo_X_features.csv')):
    X = pd.read_csv(os.path.join(processed_dir, 'geo_X_features.csv'), index_col=0)
    y = pd.read_csv(os.path.join(processed_dir, 'geo_y_target.csv'), index_col=0)['target']
    print(f'Features: {X.shape}')
    print(f'Target distribution: {y.value_counts().to_dict()}')
else:
    print('Processed data not found. Run notebook 02 or reproduce.sh first.')

---
## Build Pipelines with StabilitySelector

Each pipeline chains:
1. `StandardScaler` — z-score normalize each gene
2. `VarianceThreshold` — remove near-zero-variance genes
3. `StabilitySelector` — bootstrap 100x, keep top-K features in >=50% of resamples
4. Classifier — LR, RF, XGBoost, or MLP

All inside a single sklearn Pipeline -> preprocessing is refit on every training fold independently.

In [ ]:
FEATURE_SELECT_K = 500

def build_pipeline(model_builder):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('variance', VarianceThreshold(threshold=0.01)),
        ('feature_select', StabilitySelector(
            k=FEATURE_SELECT_K, n_bootstrap=50, min_pct=0.5, n_jobs=-1)),
        ('classifier', model_builder()),
    ])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Features after leakage removal: {X_train.shape[1]}')

---
## Train Models & Evaluate

For demonstration, we fit each model on the train split and evaluate on the holdout test split.
Full nested CV results from the production pipeline are loaded below.

In [ ]:
results = []
for name, builder in MODEL_BUILDERS.items():
    pipe = build_pipeline(builder)
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = pipe.predict(X_test)
    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    n_feat = pipe.named_steps['feature_select'].get_support().sum()
    results.append({'Model': name, 'Holdout AUC': f'{auc:.4f}',
                    'Holdout Acc': f'{acc:.4f}', 'Features Selected': n_feat})
    print(f'{name:30s} AUC={auc:.4f}  Acc={acc:.4f}  Features={n_feat}')

pd.DataFrame(results)

---
## Actual Nested Cross-Validation Results

Results from the full production pipeline (`python -m src.train --dataset geo`) with:
- 5-fold outer StratifiedKFold CV
- GridSearchCV with 3-fold inner CV for hyperparameter tuning
- StabilitySelector (B=100, p=0.5) inside every fold
- Held-out test split (20%) for final evaluation

In [ ]:
cv_results = pd.read_csv('../results/all_models_geo_leakage_fixed_metrics.csv')
cv_results.style.background_gradient(cmap='viridis', subset=['Holdout_ROC_AUC', 'CV_ROC_AUC_Mean'])

## Ablation: StabilitySelector vs SelectKBest

StabilitySelector matches or beats standard SelectKBest on all 4 models.

In [ ]:
ablation = pd.read_csv('../results/ablation_study.csv')
ablation_pivot = ablation.pivot_table(
    index='Model', columns='Selector', values='Holdout_AUC')
ablation_pivot['Delta (SS - SKB)'] = (
    ablation_pivot.iloc[:, 1].astype(float) - ablation_pivot.iloc[:, 0].astype(float))
ablation_pivot.style.background_gradient(cmap='RdYlGn', subset=['Delta (SS - SKB)'])

---
## ISEF Takeaway: Why These Models?

| Model | Strength | Holdout AUC | Why include? |
|-------|----------|-------------|--------------|
| Logistic Regression | Interpretable, calibrated baseline | 0.994 | Available in all clinical settings |
| Random Forest | Handles non-linearity, feature interactions | 0.988 | Best accuracy, best calibration |
| XGBoost | Gradient boosting, state-of-the-art | 0.991 | Highest raw AUC |
| Neural Network (MLP) | Deep learning benchmark | 0.992 | Tests if complexity helps |

**All 4 models achieve >0.97 AUC.** StabilitySelector adds robustness without sacrificing performance.
The ensemble of all 4 models reaches 0.978 AUC on external cross-platform validation.

**Next:** Full evaluation including external validation on TCGA/CPTAC, calibration benchmark with 5 methods,
survival analysis, and drug sensitivity screen (notebook 04).